# SUMMARY

**Stack**: PyTorch, SkLearn, catboost, nltk, gensim, Pandas

**Description**: Building different text classifiers
* the TfIdf + LogisticRegression option has been implemented
* the TfIdf + RandomForestClassifier option has been implemented
* the word2vec/doc2vec + linear models option has been implemented
* the word2vec/doc2vec + LSTM option has been implemented

# Start

# Данные

https://www.kaggle.com/datasets/ankurzing/sentiment-analysis-for-financial-news

## Imports and constants

In [1]:
!pip install gensim
!pip install chardet
!pip install catboost

import re
import nltk
import torch
import torch.nn as nn
import string
import chardet
import numpy as np
import pandas as pd
import multiprocessing

from tqdm import tqdm
from catboost import CatBoostClassifier
from gensim.models import Word2Vec
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from torch.utils.data import TensorDataset, DataLoader

nltk.download('stopwords')
nltk.download('punkt_tab')      
nltk.download('wordnet')    
nltk.download('omw-1.4') 
nltk.download('averaged_perceptron_tagger_eng')

RANDOM_STATE = 42
PATH_TO_DATA = "/kaggle/input/sentiment-analysis-for-financial-news/all-data.csv"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 40.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
tsfresh 0.21.0 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
imbalanced-learn 0.13.0 requires scikit-learn<2,>=1.3.2, but you have scikit-learn 1.2.2 which is incompatible.
plotnine 0.14.5 requires matplotlib>=3.8.0, but you have matplotlib 3.7.2 which is incompat

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


## Reading and analysing data

In [2]:
# open data
with open(PATH_TO_DATA, 'rb') as f:
  rawdata = f.read(50000)  # проанализировать первые 50KB
  result = chardet.detect(rawdata)

print(result['encoding'])  # показать предполагаемую кодировку
data = pd.read_csv(PATH_TO_DATA, encoding=result['encoding'])

# rename columns
data.columns = ["sentiment", "text"]
print(data.shape)

# Статистика по оценкам настроения
print(data["sentiment"].value_counts())

# Информация о длине текстов
data["text_length"] = data["text"].apply(len)
print(data["text_length"].describe())
data.head(5)

ISO-8859-1
(4845, 2)
sentiment
neutral     2878
positive    1363
negative     604
Name: count, dtype: int64
count    4845.000000
mean      128.132301
std        56.532012
min         9.000000
25%        84.000000
50%       119.000000
75%       163.000000
max       315.000000
Name: text_length, dtype: float64


,sentiment,text,text_length
0,neutral,Technopolis plans to develop in stages an area...,190
1,negative,The international electronic industry company ...,228
2,positive,With the new production plant the company woul...,206
3,positive,According to the company 's updated strategy f...,203
4,positive,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...,178


## Text processing

In [4]:
# Перевод текстов в нижний регистр
data["text"] = data["text"].apply(lambda x: x.lower())

# Delete punctuation
data["text"] = data["text"].apply(lambda x: "".join(ch for ch in x if ch not in string.punctuation))

# Delete stopwords
stop_words = set(stopwords.words("english"))
data["tokens"] = data["text"].apply(word_tokenize)
data["tokens"] = data["tokens"].apply(lambda x: [word for word in x if word not in stop_words])

# Лемматизация
lemmatizer = WordNetLemmatizer()
data["tokens"] = data["tokens"].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

# Замена чисел на специальный токен
## Функция для замены
### Плохо работает, остаются токены типа "10000odd"
def replace_numbers(token):
    if re.match(r'^-?\d+([\.,]\d+)?$', token):  # Регулярка для чисел
        return '[NUM]'
    return token

data["tokens"] = data["tokens"].apply(lambda x: [replace_numbers(token) for token in x])
data["tokens"]


0       [technopolis, plan, develop, stage, area, less...
1       [international, electronic, industry, company,...
2       [new, production, plant, company, would, incre...
3       [according, company, updated, strategy, year, ...
4       [financing, aspocomp, growth, aspocomp, aggres...
                              ...                        
4840    [london, marketwatch, share, price, ended, low...
4841    [rinkuskiai, beer, sale, fell, [NUM], per, cen...
4842    [operating, profit, fell, eur, [NUM], mn, eur,...
4843    [net, sale, paper, segment, decreased, eur, [N...
4844    [sale, finland, decreased, [NUM], january, sal...
Name: tokens, Length: 4845, dtype: object

# TfIdf + LogisticRegression

## Разделение на train, test и val и векторизация

In [4]:
data["text"] = data["tokens"].apply(lambda x: " ".join(x))

X_train, X_other, y_train, y_other = train_test_split(data["text"], data["sentiment"], test_size=0.3, random_state=RANDOM_STATE)
X_test, X_val, y_test, y_val = train_test_split(X_other, y_other, test_size=0.5, random_state=RANDOM_STATE)

vectorizer = TfidfVectorizer(min_df=5)
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)
X_val = vectorizer.transform(X_val)

## Выбор модели, обучение и оценка

In [ ]:
parameters = {
    "C": np.arange(0, 1, 0.05), 
    "penalty": ["l1", "l2"],
    "max_iter": [1000, 2000],
    "class_weight": [None, "balanced"]
}

model = LogisticRegression(multi_class="multinomial")
searcher = GridSearchCV(
    estimator=model, 
    param_grid=parameters,
    scoring="f1_macro",
    cv=10,
    n_jobs=-1,
    verbose=1
)

searcher.fit(X_train, y_train)
best_model = searcher.best_estimator_
y_pred = best_model.predict(X_test)

print("Лучшие параметры:", searcher.best_params_)
print("\nОтчет на test наборе:")
print(classification_report(y_test, y_pred))

# TfIdf + RandomForestClassifier

In [5]:
parameters = {
    'n_estimators': [100, 200],      # Количество деревьев
    'max_depth': [10, 20],       # Максимальная глубина деревьев
    'min_samples_split': [2, 5],       # Минимальное количество samples для разделения
    'min_samples_leaf': [1, 2],         # Минимальное количество samples в листе
    'max_features': ['sqrt'], # Количество признаков для разделения
    'bootstrap': [True],            # Использование бутстрэпа
    'class_weight': ['balanced']     # Балансировка классов
}

random_forest = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

searcher = GridSearchCV(
    estimator=random_forest,
    param_grid=parameters,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1,
    verbose=2
)

searcher.fit(X_train, y_train)
best_model = searcher.best_estimator_
y_pred = best_model.predict(X_test)

print("Лучшие параметры:", searcher.best_params_)
print("\nОтчет на test наборе:")
print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Лучшие параметры: {'bootstrap': True, 'class_weight': 'balanced', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}

Отчет на test наборе:
              precision    recall  f1-score   support

    negative       0.51      0.57      0.54        90
     neutral       0.73      0.88      0.80       426
    positive       0.70      0.39      0.50       211

    accuracy                           0.70       727
   macro avg       0.65      0.61      0.61       727
weighted avg       0.70      0.70      0.68       727

[CV] END bootstrap=True, class_weight=balanced, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   1.1s
[CV] END bootstrap=True, class_weight=balanced, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   0.5s
[CV] END bootstrap=True, class_weight=bala

# word2vec/doc2vec + linear models

## Разделение на train, test и val и векторизация

In [4]:
def doc_to_vec(vectorizer, words):
    vectors = [vectorizer.wv[word] for word in words if word in vectorizer.wv]
    return np.mean(vectors, axis=0) if vectors else np.zeros(vectorizer.vector_size)

# encoder = LabelEncoder()
# data["encoding"] = encoder.fit_transform(data["sentiment"])

X_train, X_other, y_train, y_other = train_test_split(data["tokens"], data["sentiment"], test_size=0.3, random_state=RANDOM_STATE)
X_test, X_val, y_test, y_val = train_test_split(X_other, y_other, test_size=0.5, random_state=RANDOM_STATE)

vectorizer = Word2Vec(
    sentences=X_train,
    vector_size=300,
    window=5, 
    min_count=5, 
    workers=4, 
    sg=1
)

X_train = np.array([doc_to_vec(vectorizer, tokens) for tokens in X_train])
X_test = np.array([doc_to_vec(vectorizer, tokens) for tokens in X_test])
X_val = np.array([doc_to_vec(vectorizer, tokens) for tokens in X_val])

In [5]:
encoder = LabelEncoder()
data["encoding"] = encoder.fit_transform(data["sentiment"])

processed_text  = [TaggedDocument(tokens, [i]) for i, tokens in enumerate(data["tokens"])]

X_train, X_other, y_train, y_other = train_test_split(processed_text, data["encoding"], test_size=0.3, random_state=RANDOM_STATE)
X_test, X_val, y_test, y_val = train_test_split(X_other, y_other, test_size=0.5, random_state=RANDOM_STATE)

vectorizer = Doc2Vec(
    vector_size=300,
    window=5,
    min_count=5,
    workers=multiprocessing.cpu_count(),
    epochs=40,
    alpha=0.025,
    min_alpha=0.00025,
    dm=1
)

vectorizer.build_vocab(X_train)
vectorizer.train(
    X_train,
    total_examples=vectorizer.corpus_count,
    epochs=vectorizer.epochs
)

X_train = torch.tensor([vectorizer.dv[doc.tags[0]] for doc in X_train])
X_test = torch.tensor([vectorizer.infer_vector(doc.words) for doc in X_test])
X_val = torch.tensor([vectorizer.infer_vector(doc.words) for doc in X_val])

y_train = torch.tensor(y_train.values)
y_test = torch.tensor(y_test.values)
y_val = torch.tensor(y_val.values)

/tmp/ipykernel_36/3181347873.py:27: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  X_train = torch.tensor([vectorizer.dv[doc.tags[0]] for doc in X_train])


## Обучение

### SVM Classificator

In [ ]:
parameters = {
    "C": [0.1, 1, 10, 100],
    "class_weight": [None, "balanced"]
}

svc = SVC(random_state=RANDOM_STATE)
searcher = GridSearchCV(
    estimator=svc,
    param_grid=parameters,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

searcher.fit(X_train, y_train)
best_model = searcher.best_estimator_
y_pred = best_model.predict(X_test)

print("Лучшие параметры:", searcher.best_params_)
print("\nОтчет на test наборе:")
print(classification_report(y_test, y_pred))

### Random Forest

In [6]:
parameters = {
    'n_estimators': [100, 200],      # Количество деревьев
    'max_depth': [10, 20],       # Максимальная глубина деревьев
    'min_samples_split': [2, 5],       # Минимальное количество samples для разделения
    'min_samples_leaf': [1, 2],         # Минимальное количество samples в листе
    'max_features': ['sqrt'], # Количество признаков для разделения
    'bootstrap': [True],            # Использование бутстрэпа
    'class_weight': ['balanced']     # Балансировка классов
}

random_forest = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

searcher = GridSearchCV(
    estimator=random_forest,
    param_grid=parameters,
    scoring='f1_macro',
    cv=3,
    n_jobs=-1,
    verbose=2
)

searcher.fit(X_train, y_train)
best_model = searcher.best_estimator_
y_pred = best_model.predict(X_test)

print("Лучшие параметры:", searcher.best_params_)
print("\nОтчет на test наборе:")
print(classification_report(y_test, y_pred))

Fitting 3 folds for each of 16 candidates, totalling 48 fits
Лучшие параметры: {'bootstrap': True, 'class_weight': 'balanced', 'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 5, 'n_estimators': 100}

Отчет на test наборе:
              precision    recall  f1-score   support

    negative       0.32      0.11      0.17        90
     neutral       0.64      0.86      0.73       426
    positive       0.45      0.26      0.33       211

    accuracy                           0.59       727
   macro avg       0.47      0.41      0.41       727
weighted avg       0.54      0.59      0.54       727

[CV] END bootstrap=True, class_weight=balanced, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   8.2s
[CV] END bootstrap=True, class_weight=balanced, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   4.2s
[CV] END bootstrap=True, class_weight=bala

### catboost

In [7]:
parameters = {
    "learning_rate": [0.01, 0.05, 0.1],
    "l2_leaf_reg": [1, 3, None],
    "iterations": [500, 1000, 1500]
}

catboost = CatBoostClassifier(
    depth=6,
    loss_function='MultiClass',
    random_seed=RANDOM_STATE,
    verbose=0
)

searcher = GridSearchCV(
    estimator=catboost,
    param_grid=parameters,
    scoring='f1_macro',
    cv=3,
    n_jobs=1,
)

searcher.fit(X_train, y_train)
best_model = searcher.best_estimator_
y_pred = best_model.predict(X_test)

print("Лучшие параметры:", searcher.best_params_)
print("\nОтчет на test наборе:")
print(classification_report(y_test, y_pred))

Лучшие параметры: {'iterations': 1500, 'l2_leaf_reg': 1, 'learning_rate': 0.1}

Отчет на test наборе:
              precision    recall  f1-score   support

    negative       0.76      0.50      0.60        90
     neutral       0.75      0.90      0.81       426
    positive       0.67      0.49      0.57       211

    accuracy                           0.73       727
   macro avg       0.73      0.63      0.66       727
weighted avg       0.73      0.73      0.72       727



### LSTM

In [10]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, dropout_rate=0.3):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout_rate)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x, h0=None, c0=None):
        x = x.unsqueeze(1)
        
        if h0 is None or c0 is None:
            h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
            c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        
        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.dropout(out[:, -1, :])
        out = self.fc(out)
        return out

    def predict_proba(self, x):
        logits = self.forward(x)
        return self.softmax(logits)


In [20]:
input_dim = 300   # Размерность вектора слов (word embedding dimension)
hidden_dim = 128  # Размерность скрытого состояния LSTM
layer_dim = 2     # Количество LSTM слоев
output_dim = 3    # Количество классов: positive, negative, neutral

dataset = TensorDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

model = LSTMModel(input_dim, hidden_dim, layer_dim, output_dim)

# Определение функции потерь и оптимизатора
criterion = nn.CrossEntropyLoss()  # Для многоклассовой классификации
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Пример обучения
for epoch in tqdm(range(100)):
    for batch_x, batch_y in dataloader:
        # Forward pass
        outputs = model(batch_x)
        
        # Calculate loss
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

100%|██████████| 100/100 [01:19<00:00,  1.26it/s]


In [21]:
# Предсказание
with torch.no_grad():
    outputs = model(X_test)
    _, predicted = torch.max(outputs.data, 1)
    probabilities = model.predict_proba(X_test)
    print(classification_report(y_test, predicted))

              precision    recall  f1-score   support

           0       0.32      0.48      0.38        90
           1       0.73      0.57      0.64       426
           2       0.44      0.54      0.48       211

    accuracy                           0.55       727
   macro avg       0.50      0.53      0.50       727
weighted avg       0.59      0.55      0.56       727

